In [ ]:
import json

from data.openalex import openalex
from data.scopus import scopus
#from utils import *
#from journals import *
#from duplicates import *
#from citescore import *
#from merge import *
#from rti import *
#from ef import *
#from snip import *
#from top import *
#from alt import *
#from rank import *

### OpenAlex

Data:
* id
* issns
* title
* other titles
* publisher id
* publisher title
* publisher other titles
* link

Metrics:
* Impact Factor (IF)
* H index

In [ ]:
publishers, journals = openalex.get_data()

### Scopus

Data:
* scopus id
* other titles
* active
* last year
* publisher other titles
* fields

In [ ]:
data = scopus.get_data()

### SCImago

* Impact Factor (IF)
* H index
* SCImago Journal Rank (SJR)

In [ ]:
journals, metrics = scimago.get_data()

### Journals

* Impact Factor (IF)
* H index
* SCImago Journal Rank (SJR)

In [ ]:
journals, metrics = get_journals()

In [ ]:
journals = clean_duplicates(journals, metrics)

* CiteScore
* Source Normalized Impact per Paper (SNIP)
* SCImago Journal Rank (SJR)

In [ ]:
data_journals, data_metrics = get_citescore()

In [ ]:
exact_matches, pairs = create_pairs(journals, metrics, data_journals)

In [ ]:
exact_matches = include_data(journals, metrics, data_journals, exact_matches, pairs)

In [ ]:
metrics = update_metrics(metrics, data_metrics, exact_matches)

* Rigor & Transparency Index

In [ ]:
data_journals, data_metrics = get_rti()

In [ ]:
exact_matches, pairs = create_pairs(journals, metrics, data_journals)

In [ ]:
exact_matches = include_data(journals, metrics, data_journals, exact_matches, pairs)

In [ ]:
metrics = update_metrics(metrics, data_metrics, exact_matches)

* Eigenfactor Score (EF)
* Article Influence Score (AI)

In [ ]:
data_journals, data_metrics = get_ef()

In [ ]:
exact_matches, pairs = create_pairs(journals, metrics, data_journals)

In [ ]:
exact_matches = include_data(journals, metrics, data_journals, exact_matches, pairs)

In [ ]:
metrics = update_metrics(metrics, data_metrics, exact_matches)

* Source Normalized Impact per Paper (SNIP)
* Self-Citation Ratio

In [ ]:
data_journals, data_metrics = get_snip()

In [ ]:
exact_matches, pairs = create_pairs(journals, metrics, data_journals)

In [ ]:
exact_matches = include_data(journals, metrics, data_journals, exact_matches, pairs)

In [ ]:
metrics = update_metrics(metrics, data_metrics, exact_matches)

* Transparency and Openness Promotion Factor (TOP Factor)

In [ ]:
data_journals, data_metrics = get_top()

In [ ]:
exact_matches, pairs = create_pairs(journals, metrics, data_journals)

In [ ]:
exact_matches = include_data(journals, metrics, data_journals, exact_matches, pairs)

In [ ]:
metrics = update_metrics(metrics, data_metrics, exact_matches)

* Altmetrics

In [ ]:
data_journals, data_metrics = get_alt()

In [ ]:
exact_matches, pairs = create_pairs(journals, metrics, data_journals)

In [ ]:
exact_matches = include_data(journals, metrics, data_journals, exact_matches, pairs)

In [ ]:
metrics = update_metrics(metrics, data_metrics, exact_matches)

Fixes

In [ ]:
metrics['if_1']['21100200832'] = {'year': 2017, 'value': 0.667}
journals['23571']['scopes'] = ['Multidisciplinary']
journals['85291']['titles'] = ['Journal of the American Medical Association (JAMA)'] + journals['85291']['titles']
journals['21206']['titles'] = [journals['21206']['titles'][0]]
journals['23571']['titles'] = [journals['23571']['titles'][0]]

Merge metrics

In [ ]:
metrics = merge_metrics(journals, metrics)

Save

In [ ]:
with open('./raw/homepages.json', 'r', encoding='utf-8') as file:
	homepages = json.load(file)

homepages['21206'] = 'https://www.nature.com'
homepages['23571'] = 'https://www.science.org/journal/science'

result = []

for id, journal in journals.items():
	result.append({
		'id': id,
		'titles': journal['titles'],
		'issns': journal['issns'],
		'link': homepages.get(id),
		'type': journal['type'],
		'publisher': journal['publishers'][0] if len(journal['publishers']) > 0 else None,
		'scopes': journal['scopes'],
		'metrics': {
			'h': metrics['h'].get(id, {'value': None})['value'],
			'if': metrics['if'].get(id, {'value': None})['value'],
			'cs': metrics['cs'].get(id, {'value': None})['value'],
			'sjr': metrics['sjr'].get(id, {'value': None})['value'],
			'snip': metrics['snip'].get(id, {'value': None})['value'],
			'ef': metrics['ef'].get(id, {'value': None})['value'],
			'ai': metrics['ai'].get(id, {'value': None})['value'],
			'self': metrics['self'].get(id, {'value': None})['value'],
			'rti': metrics['rti'].get(id, {'value': None})['value'],
			'top': metrics['top'].get(id, {'value': None})['value'],
			'alt': metrics['alt'].get(id, {'value': None})['value']
		}
	})

result.sort(key=lambda x: x['titles'][0])

with open('raw_data.json', 'w', encoding='utf-8') as file:
	json.dump({
		'scopes': SCOPES,
		'journals': result,
	}, file, ensure_ascii=False)